# Angle processor

Batch-converts larva position files into files enriched with movement angles and
step distances.

Input:  `larva_positions_{id}.csv` (columns: Frame, X, Y in pixels)
Output: `larva_positions_{id}_with_angles.csv` (columns: Frame, X, Y, Angle, Distance in mm)


## 1. Imports

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Optional, List, Dict
from IPython.display import display, HTML

## 2. Configuration and constants

In [ ]:
# ============================================================================
# CONFIGURATION CONSTANTS
# ============================================================================

SCALE = 0.2  # mm/px - pixel-to-millimeter conversion factor

# Base paths
BASE_TRACKING_PATH = Path("tracking")

In [ ]:
# ============================================================================
# MAPPING DICTIONARIES
# ============================================================================

SESSIONS = {
    1: "01",
    2: "02"
}

# Experiments for session 01
EXPERIMENTS_01 = {
    1: "control",
    2: "pbs",
    3: "coli_2x10_8",
    4: "coli_5x_10_7",
    5: "coli_5x_10_8"
}

# Experiments for session 02
EXPERIMENTS_02 = {
    1: "control",
    2: "pbs",
    3: "coli_2x10_8",
    4: "coli_5x10_7",
    5: "coli_5x10_8"
}

# Mapping of experiment names to display names
EXPERIMENT_DISPLAY_NAMES = {
    "control": "Control (no injection)",
    "naklucie": "Injection",
    "pbs": "PBS",
    "coli_2x10_8": "E. coli 2x10^8",
    "coli_5x_10_7": "E. coli 5x10^7",
    "coli_5x_10_8": "E. coli 5x10^8",
    "coli_5x10_7": "E. coli 5x10^7",
    "coli_5x10_8": "E. coli 5x10^8"
}

# Number of larvae per experiment
LARVAE_PER_EXPERIMENT = 6

## 3. Data-selection functions

In [ ]:
# ============================================================================
# DATA-SELECTION FUNCTIONS
# ============================================================================

def display_sessions():
    """Print the available sessions."""
    print("\n=== Available sessions ===")
    for num, name in SESSIONS.items():
        print(f"{num}. Session {name}")
    print("==========================\n")

def display_experiments(session: str):
    """Print the available experiments for a session."""
    experiments = EXPERIMENTS_01 if session == "01" else EXPERIMENTS_02
    print(f"\n=== Experiments for session {session} ===")
    for num, name in experiments.items():
        display_name = EXPERIMENT_DISPLAY_NAMES.get(name, name)
        print(f"{num}. {name} ({display_name})")
    print("=" * 40 + "\n")

def get_session() -> str:
    """Read the session number from the user."""
    display_sessions()
    while True:
        try:
            session_num = int(input('Enter session number (1-2): '))
            if session_num in SESSIONS:
                return SESSIONS[session_num]
            else:
                print('Error: choose 1 or 2')
        except ValueError:
            print('Error: enter an integer')

def get_experiment(session: str) -> str:
    """Read the experiment from the user."""
    experiments = EXPERIMENTS_01 if session == "01" else EXPERIMENTS_02
    display_experiments(session)
    while True:
        try:
            exp_num = int(input(f'Enter experiment number (1-{len(experiments)}): '))
            if exp_num in experiments:
                return experiments[exp_num]
            else:
                print(f'Error: choose a number between 1 and {len(experiments)}')
        except ValueError:
            print('Error: enter an integer')

def get_larva_selection() -> Optional[int]:
    """Read the larva selection from the user (0 = all)."""
    print("\n=== Larva selection ===")
    print("0. All larvae (batch processing)")
    for i in range(1, LARVAE_PER_EXPERIMENT + 1):
        print(f"{i}. Larva {i}")
    print("=" * 25 + "\n")

    while True:
        try:
            larva_num = int(input(f'Enter larva number (0 = all, 1-{LARVAE_PER_EXPERIMENT}): '))
            if larva_num == 0:
                return None  # all larvae
            elif 1 <= larva_num <= LARVAE_PER_EXPERIMENT:
                return larva_num
            else:
                print(f'Error: choose 0 or a number between 1 and {LARVAE_PER_EXPERIMENT}')
        except ValueError:
            print('Error: enter an integer')

## 4. AngleProcessor class

In [ ]:
# ============================================================================
# ANGLE PROCESSOR CLASS
# ============================================================================

class AngleProcessor:
    """
    Converts larva position files into files with movement angles and distances.

    Transformation:
    - scale coordinates from px to mm (SCALE = 0.2 mm/px)
    - compute movement angles (0-360 deg, 0 = N, clockwise)
    - compute step distances between consecutive frames
    """

    def __init__(self, session: str, experiment: str, scaling_factor: float = SCALE):
        self.session = session
        self.experiment = experiment
        self.scaling_factor = scaling_factor
        self.display_name = EXPERIMENT_DISPLAY_NAMES.get(experiment, experiment)
        self.tracking_dir = BASE_TRACKING_PATH / session / experiment

        # Processing statistics
        self.stats: Dict[int, Dict] = {}

    def _get_input_file(self, larva_num: int) -> Optional[Path]:
        """Return the input file path (the basic position file)."""
        # Prefer files without angles (raw data in px)
        file_options = [
            self.tracking_dir / f'larva_positions_{larva_num}.csv',
            self.tracking_dir / f'larva_positions_{larva_num}_cleaned.csv',
        ]

        for file_path in file_options:
            if file_path.exists():
                return file_path

        return None

    def _get_output_file(self, larva_num: int) -> Path:
        """Return the output file path."""
        return self.tracking_dir / f'larva_positions_{larva_num}_with_angles.csv'

    def _detect_data_unit(self, df: pd.DataFrame) -> str:
        """
        Detect whether data is in pixels or millimeters.

        Returns:
            'px' or 'mm'
        """
        if len(df) == 0:
            return 'px'

        max_val = max(df['X'].max(), df['Y'].max())

        # A dish is ~95 mm ~= 475 px; values > 200 are certainly pixels
        return 'px' if max_val > 200 else 'mm'

    def process_larva(self, larva_num: int, overwrite: bool = False) -> bool:
        """
        Process a single larva's position file.

        Args:
            larva_num: larva number (1-6)
            overwrite: whether to overwrite existing files

        Returns:
            True on success, False on error.
        """
        input_file = self._get_input_file(larva_num)
        output_file = self._get_output_file(larva_num)

        if input_file is None:
            self.stats[larva_num] = {
                'status': 'missing',
                'message': 'No input file'
            }
            return False

        if output_file.exists() and not overwrite:
            self.stats[larva_num] = {
                'status': 'skipped',
                'message': 'File already exists (use overwrite=True)'
            }
            return False

        try:
            df = pd.read_csv(input_file)
            original_count = len(df)

            data_unit = self._detect_data_unit(df)

            # Scale coordinates only if the data is in pixels
            if data_unit == 'px':
                df['X'] = (df['X'] * self.scaling_factor).round(2)
                df['Y'] = (df['Y'] * self.scaling_factor).round(2)

            # Compute angles and distances
            dx = df['X'].diff()
            dy = df['Y'].diff()

            # Angle: 0 deg = N, increases clockwise (compass-style)
            angles = np.degrees(np.arctan2(-dx, dy))
            angles_clockwise = (360 - angles) % 360

            distance = np.sqrt(dx**2 + dy**2)

            df['Angle'] = angles_clockwise.round().astype('Int64')
            df['Distance'] = distance.round(2)

            # Drop the first NaN row (from diff)
            df.dropna(subset=['Angle', 'Distance'], inplace=True)

            df.to_csv(output_file, index=False)

            self.stats[larva_num] = {
                'status': 'success',
                'input_file': input_file.name,
                'output_file': output_file.name,
                'original_count': original_count,
                'processed_count': len(df),
                'data_unit': data_unit,
                'message': f'Processed {len(df)} records'
            }
            return True

        except Exception as e:
            self.stats[larva_num] = {
                'status': 'error',
                'message': str(e)
            }
            return False

    def process_all_larvae(self, overwrite: bool = False) -> Dict[int, bool]:
        """
        Process the position files for all larvae in the experiment group.

        Returns:
            Dict: {larva_num: success}
        """
        results = {}
        for larva_num in range(1, LARVAE_PER_EXPERIMENT + 1):
            results[larva_num] = self.process_larva(larva_num, overwrite)
        return results

    def print_summary(self):
        """Print the processing summary."""
        print("\n" + "=" * 60)
        print(f"PROCESSING SUMMARY")
        print(f"   Session: {self.session}")
        print(f"   Experiment: {self.experiment} ({self.display_name})")
        print(f"   Directory: {self.tracking_dir}")
        print("=" * 60)

        success_count = 0
        skip_count = 0
        error_count = 0

        for larva_num in sorted(self.stats.keys()):
            stat = self.stats[larva_num]
            status = stat['status']

            if status == 'success':
                icon = '[ok]'
                success_count += 1
                details = f"{stat['input_file']} -> {stat['output_file']} ({stat['processed_count']} records)"
            elif status == 'skipped':
                icon = '[skip]'
                skip_count += 1
                details = stat['message']
            elif status == 'missing':
                icon = '[warn]'
                error_count += 1
                details = stat['message']
            else:
                icon = '[x]'
                error_count += 1
                details = stat['message']

            print(f"  {icon} Larva {larva_num}: {details}")

        print("\n" + "-" * 60)
        print(f"  Success: {success_count} | Skipped: {skip_count} | Errors: {error_count}")
        print("=" * 60 + "\n")

## 5. Data selection - RUN THIS CELL

In [ ]:
# ============================================================================
# DATA SELECTION - RUN THIS CELL
# ============================================================================

SESSION = get_session()
EXPERIMENT = get_experiment(SESSION)
LARVA_SELECTION = get_larva_selection()  # None = all

print("\n" + "=" * 50)
print("SELECTED:")
print(f"  Session: {SESSION}")
print(f"  Experiment: {EXPERIMENT}")
print(f"  Name: {EXPERIMENT_DISPLAY_NAMES.get(EXPERIMENT, EXPERIMENT)}")
if LARVA_SELECTION is None:
    print(f"  Larvae: ALL (1-{LARVAE_PER_EXPERIMENT})")
else:
    print(f"  Larva: {LARVA_SELECTION}")
print(f"  Directory: tracking/{SESSION}/{EXPERIMENT}/")
print("=" * 50)

## 6. Processing - RUN THIS CELL

In [ ]:
# ============================================================================
# PROCESSING
# ============================================================================

OVERWRITE = False  # set to True to overwrite existing files

processor = AngleProcessor(SESSION, EXPERIMENT)

print(f"\nStarting processing...")
print(f"   Overwrite: {'YES' if OVERWRITE else 'NO'}")
print()

if LARVA_SELECTION is None:
    results = processor.process_all_larvae(overwrite=OVERWRITE)
else:
    results = {LARVA_SELECTION: processor.process_larva(LARVA_SELECTION, overwrite=OVERWRITE)}

processor.print_summary()

## 7. Batch processing - all experiment groups

Optional section to process all files in a session at once.


In [ ]:
# ============================================================================
# BATCH PROCESSING - ALL GROUPS IN A SESSION
# ============================================================================

def process_entire_session(session: str, overwrite: bool = False):
    """
    Process all experiment groups in a session.

    Args:
        session: session number ("01" or "02")
        overwrite: whether to overwrite existing files
    """
    experiments = EXPERIMENTS_01 if session == "01" else EXPERIMENTS_02

    print("\n" + "#" * 70)
    print(f"# BATCH PROCESSING - SESSION {session}")
    print(f"# Experiment groups: {len(experiments)}")
    print(f"# Overwrite: {'YES' if overwrite else 'NO'}")
    print("#" * 70)

    total_success = 0
    total_skipped = 0
    total_errors = 0

    for exp_num, experiment in experiments.items():
        print(f"\n{'='*60}")
        print(f"[{exp_num}/{len(experiments)}] {experiment} ({EXPERIMENT_DISPLAY_NAMES.get(experiment, experiment)})")
        print("="*60)

        processor = AngleProcessor(session, experiment)
        results = processor.process_all_larvae(overwrite=overwrite)

        for larva_num, stat in processor.stats.items():
            status = stat['status']
            if status == 'success':
                total_success += 1
                print(f"  [ok] Larva {larva_num}: OK")
            elif status == 'skipped':
                total_skipped += 1
                print(f"  [skip] Larva {larva_num}: skipped")
            else:
                total_errors += 1
                print(f"  [x] Larva {larva_num}: {stat['message']}")

    print("\n" + "#" * 70)
    print(f"# SESSION {session} SUMMARY")
    print(f"#   Success: {total_success}")
    print(f"#   Skipped: {total_skipped}")
    print(f"#   Errors: {total_errors}")
    print("#" * 70 + "\n")

In [ ]:
# Uncomment to process the whole session 01
# process_entire_session("01", overwrite=False)

In [ ]:
# Uncomment to process the whole session 02
# process_entire_session("02", overwrite=False)

In [ ]:
# Uncomment to process BOTH sessions
# process_entire_session("01", overwrite=False)
# process_entire_session("02", overwrite=False)